# Pandas DataFrame — Session 17

In the previous session you mastered the Series — a one-dimensional labeled array that behaves like a single Excel column with row numbers on the left and values on the right. Now the picture widens to its natural companion: the **DataFrame**, the two-dimensional labeled table that surveys every data science project in this course. If a Series is one column, a DataFrame is the whole spreadsheet: many columns side by side, all sharing the same row labels. Under the hood each column of a DataFrame is itself a Series, so everything you learned about Series — labels, dtypes, methods, filtering — carries straight over, column by column.

The DataFrame is also the bridge between pandas and the real world. Most data arrives as CSV files, and `pd.read_csv(...)` turns any such file into a DataFrame in one line. From there you can inspect its shape, its column types, its missing values, and its summary statistics, then select columns, select rows, and combine the two to cut out exactly the window of data you need.

Here is the roadmap for this session:

1. Creating a DataFrame from a list of lists, from a dictionary of lists, and directly from a CSV file.
2. Inspecting a DataFrame with shape, dtypes, index, columns, and values, plus the info() and describe() summaries.
3. Running maths across rows and columns with the axis parameter — the idea that sums and means can go down OR across.
4. Selecting one column into a Series, or several columns into a new DataFrame.
5. Selecting rows by integer position with iloc and by index label with loc.
6. Selecting rows and columns at the same time with two-dimensional indexing.
7. Filtering rows with boolean conditions — the pandas equivalent of SQL WHERE.
8. Adding brand-new columns, both as constants and as values derived from existing columns.
9. Optimizing memory by converting column dtypes, including the compact category dtype.
10. Exploring categorical columns quickly with value_counts and plots.

By the end of the session you will be able to load a real dataset, take its pulse in seconds, and reach in to drag out exactly the rows and columns your question needs. The DataFrame is where pandas earns its reputation, and everything you learn here will be used — and extended — in every later pandas session, including grouping and merging.

In [ ]:
import numpy as np
import pandas as pd

# 1. Creating a DataFrame

## 1.1 From lists, dicts, and CSV

The very first cells build a table three different ways, and each tells you something about what a DataFrame really is.

```python
student_data = [[100, 80, 10], [90, 70, 7], [120, 100, 14], [80, 50, 2]]
pd.DataFrame(student_data, columns=['iq', 'marks', 'package'])
```

The list-of-lists route treats each inner list as a row. `pd.DataFrame(...)` wraps them into a table with the automatic 0-to-n row index, and the `columns=` parameter names each column — iq, marks, package. Without it, the columns would be anonymous 0, 1, 2.

The dictionary route reads more like a real table:

```python
student_dict = {'name': [...], 'iq': [...], 'marks': [...], 'package': [...]}
students = pd.DataFrame(student_dict)
```

Here every key becomes a column name and the list under it becomes that column's values. This is the preferred construction style because the column names are written right next to their data. Notice that `students.set_index('name', inplace=True)` then promotes the name column into the row index, which is what the notebook output shows — students are later looked up by their names exactly like dictionary keys.

Finally, the CSV route is the one you will use most in practice: `movies = pd.read_csv('movies.csv')` and `ipl = pd.read_csv('ipl-matches.csv')` load entire files into DataFrames in a single call, headers and all.

## 1.2 Deep dive — creating tables

Read the three construction styles as three levels of control over your table.

A list of lists gives you full positional control: you decide column order through the `columns=` list, and each inner list is one row. It is verbose to type but perfect when data comes from an algorithm or a parsed structure. The printed table shows the columns you named and rows indexed 0, 1, 2, 3.

A dictionary of lists is the most readable: the words iq, marks, and package appear once beside their data, not once per row. The code can then attach a meaningful row index with `set_index('name', inplace=True)`, converting the name column into the label axis. That single line turns the DataFrame into a lookup table where `students.loc['nitish']` retrieves a whole row by name. Because the operation runs `inplace=True`, the change is permanent on the students object — the printed output shows the name column as the left-hand index.

A list of file paths — `pd.read_csv('movies.csv')` and `pd.read_csv('ipl-matches.csv')` — is the real world. The movies file loads 1629 rows with 18 columns; the IPL file loads 950 match rows with 20 columns. In every case the result is the same structure: a two-dimensional table of rows and columns where each column is a Series sharing one index.

Whatever the source, the mental model never changes: a DataFrame is rows on one axis, columns on the other, and a name attached to every column. All the power of pandas flows from that simple contract.

In [ ]:
# using lists
student_data = [
    [100,80,10],
    [90,70,7],
    [120,100,14],
    [80,50,2]
]

pd.DataFrame(student_data,columns=['iq','marks','package'])

    iq  marks  package
0  100     80       10
1   90     70        7
2  120    100       14
3   80     50        2

In [ ]:
# using dicts

student_dict = {
    'name':['nitish','ankit','rupesh','rishabh','amit','ankita'],
    'iq':[100,90,120,80,0,0],
    'marks':[80,70,100,50,0,0],
    'package':[10,7,14,2,0,0]
}

students = pd.DataFrame(student_dict)
students.set_index('name',inplace=True)
students

          iq  marks  package
name                        
nitish   100     80       10
ankit     90     70        7
rupesh   120    100       14
rishabh   80     50        2
amit       0      0        0
ankita     0      0        0

In [ ]:
# using read_csv
movies = pd.read_csv('movies.csv')
movies

                                   title_x     imdb_id  \
0                 Uri: The Surgical Strike   tt8291224   
1                            Battalion 609   tt9472208   
2     The Accidental Prime Minister (film)   tt6986710   
3                          Why Cheat India   tt8108208   
4                          Evening Shadows   tt6028796   
...                                    ...         ...   
1624                 Tera Mera Saath Rahen   tt0301250   
1625                  Yeh Zindagi Ka Safar   tt0298607   
1626                       Sabse Bada Sukh   tt0069204   
1627                                 Daaka  tt10833860   
1628                              Humsafar   tt2403201   

                                            poster_path  \
0     https://upload.wikimedia.org/wikipedia/en/thum...   
1                                                   NaN   
2     https://upload.wikimedia.org/wikipedia/en/thum...   
3     https://upload.wikimedia.org/wikipedia/en/thum...   
4       

In [ ]:
ipl = pd.read_csv('ipl-matches.csv')
ipl

          ID        City        Date   Season  MatchNumber  \
0    1312200   Ahmedabad  2022-05-29     2022        Final   
1    1312199   Ahmedabad  2022-05-27     2022  Qualifier 2   
2    1312198     Kolkata  2022-05-25     2022   Eliminator   
3    1312197     Kolkata  2022-05-24     2022  Qualifier 1   
4    1304116      Mumbai  2022-05-22     2022           70   
..       ...         ...         ...      ...          ...   
945   335986     Kolkata  2008-04-20  2007/08            4   
946   335985      Mumbai  2008-04-20  2007/08            5   
947   335984       Delhi  2008-04-19  2007/08            3   
948   335983  Chandigarh  2008-04-19  2007/08            2   
949   335982   Bangalore  2008-04-18  2007/08            1   

                           Team1                        Team2  \
0               Rajasthan Royals               Gujarat Titans   
1    Royal Challengers Bangalore             Rajasthan Royals   
2    Royal Challengers Bangalore         Lucknow Super Giant

# 2. Attributes and Methods

## 2.1 shape, dtypes, index, columns, values

When a fresh DataFrame lands in memory, the first reflex is to interrogate its structure. The attributes in this section answer five basic questions.

`shape` returns the dimensions as a `(rows, columns)` tuple. `movies.shape` gives `(950, 20)` — wait, that is the IPL frame's shape; the movies frame reports (1629, 18). Try not to fixate on the numbers themselves and instead make shape your habitual opening move: rows first, columns second.

`dtypes` prints the dtype of every column in one go. The IPL output shows ID as int64 and nearly everything else — City, Date, Season, MatchNumber, Team1, Team2, Venue — as object, because these columns hold text. Spotting an object column where numbers should be is the fastest way to catch a parsing problem.

`index` and `columns` report the two label axes. The default index is a RangeIndex(start=0, stop=950, step=1), and `columns` returns the Index of names — `Index(['iq', 'marks', 'package'], dtype='object')` in the student frame. Knowing your columns is a prerequisite to selecting them.

`values` returns the entire table as a two-dimensional NumPy array with the labels stripped away. The IPL output shows rows of raw data — match IDs, team names, umpire names — which is exactly what you pass to a numpy-only library when you need it.

## 2.2 info, describe, isnull, duplicated, rename

Attributes report structure; methods report health. This section runs the standard doctors' checklist.

`movies.info()` prints a compact summary: the class, the RangeIndex bounds, the total column count, and a per-column table of non-null counts and dtypes. The movies frame reports 1629 non-null values in most columns and a handful of columns with gaps — a perfect first warning that cleaning will be needed. `ipl.info()` does the same for the cricket data and reveals, for example, that City has 899 non-null entries out of 950.

`movies.describe()` computes the statistical summary for every numeric column at once — count, mean, std, min, percentiles, max. The output for year_of_release, imdb_rating, and imdb_votes gives an instant sense of the data's centre and spread.

`movies.isnull().sum()` counts the missing values column by column, and `movies.duplicated().sum()` counts the duplicate rows — 0 for movies, but 1 for the students frame, meaning one student row is a full repeat.

`students.rename(columns={'marks': 'percent', 'package': 'lpa'}, inplace=True)` relabels columns before later cells use the friendlier names percent and lpa. Preview with head, tail, and sample, check health with info, describe, isnull, and duplicated, and relabel with rename — that is the entire inspection ritual, front to back.

In [ ]:
# shape
movies.shape
ipl.shape

(950, 20)

In [ ]:
# dtypes
movies.dtypes
ipl.dtypes

ID                   int64
City                object
Date                object
Season              object
MatchNumber         object
Team1               object
Team2               object
Venue               object
TossWinner          object
TossDecision        object
SuperOver           object
WinningTeam         object
WonBy               object
Margin             float64
method              object
Player_of_Match     object
Team1Players        object
Team2Players        object
Umpire1             object
Umpire2             object
dtype: object

In [ ]:
# index
movies.index
ipl.index

RangeIndex(start=0, stop=950, step=1)

In [ ]:
# columns
movies.columns
ipl.columns
student.columns

Index(['iq', 'marks', 'package'], dtype='object')

In [ ]:
# values
student.values
ipl.values

array([[1312200, 'Ahmedabad', '2022-05-29', ...,
        "['WP Saha', 'Shubman Gill', 'MS Wade', 'HH Pandya', 'DA Miller', 'R Tewatia', 'Rashid Khan', 'R Sai Kishore', 'LH Ferguson', 'Yash Dayal', 'Mohammed Shami']",
        'CB Gaffaney', 'Nitin Menon'],
       [1312199, 'Ahmedabad', '2022-05-27', ...,
        "['YBK Jaiswal', 'JC Buttler', 'SV Samson', 'D Padikkal', 'SO Hetmyer', 'R Parag', 'R Ashwin', 'TA Boult', 'YS Chahal', 'M Prasidh Krishna', 'OC McCoy']",
        'CB Gaffaney', 'Nitin Menon'],
       [1312198, 'Kolkata', '2022-05-25', ...,
        "['Q de Kock', 'KL Rahul', 'M Vohra', 'DJ Hooda', 'MP Stoinis', 'E Lewis', 'KH Pandya', 'PVD Chameera', 'Mohsin Khan', 'Avesh Khan', 'Ravi Bishnoi']",
        'J Madanagopal', 'MA Gough'],
       ...,
       [335984, 'Delhi', '2008-04-19', ...,
        "['T Kohli', 'YK Pathan', 'SR Watson', 'M Kaif', 'DS Lehmann', 'RA Jadeja', 'M Rawat', 'D Salunkhe', 'SK Warne', 'SK Trivedi', 'MM Patel']",
        'Aleem Dar', 'GA Pratapkumar'],
    

In [ ]:
# head and tail
movies.head(2)

                    title_x    imdb_id  \
0  Uri: The Surgical Strike  tt8291224   
1             Battalion 609  tt9472208   

                                         poster_path  \
0  https://upload.wikimedia.org/wikipedia/en/thum...   
1                                                NaN   

                                           wiki_link  \
0  https://en.wikipedia.org/wiki/Uri:_The_Surgica...   
1        https://en.wikipedia.org/wiki/Battalion_609   

                    title_y            original_title  is_adult  \
0  Uri: The Surgical Strike  Uri: The Surgical Strike         0   
1             Battalion 609             Battalion 609         0   

   year_of_release runtime            genres  imdb_rating  imdb_votes  \
0             2019     138  Action|Drama|War          8.4       35112   
1             2019     131               War          4.1          73   

                                               story  \
0  Divided over five chapters  the film chronicle...   
1

In [ ]:
ipl.tail(2)

         ID        City        Date   Season MatchNumber  \
948  335983  Chandigarh  2008-04-19  2007/08           2   
949  335982   Bangalore  2008-04-18  2007/08           1   

                           Team1                  Team2  \
948              Kings XI Punjab    Chennai Super Kings   
949  Royal Challengers Bangalore  Kolkata Knight Riders   

                                          Venue                   TossWinner  \
948  Punjab Cricket Association Stadium, Mohali          Chennai Super Kings   
949                       M Chinnaswamy Stadium  Royal Challengers Bangalore   

    TossDecision SuperOver            WinningTeam WonBy  Margin method  \
948          bat         N    Chennai Super Kings  Runs    33.0    NaN   
949        field         N  Kolkata Knight Riders  Runs   140.0    NaN   

    Player_of_Match                                       Team1Players  \
948      MEK Hussey  ['K Goel', 'JR Hopes', 'KC Sangakkara', 'Yuvra...   
949     BB McCullum  ['R Drav

In [ ]:
# sample
ipl.sample(5)

          ID        City        Date   Season MatchNumber  \
336  1082628      Mumbai  2017-05-01     2017          38   
98   1254107     Sharjah  2021-09-25     2021          37   
890   392182   Cape Town  2009-04-18     2009           2   
157  1216533   Abu Dhabi  2020-10-19  2020/21          37   
386   980991  Chandigarh  2016-05-15     2016          46   

                           Team1                        Team2  \
336               Mumbai Indians  Royal Challengers Bangalore   
98                  Punjab Kings          Sunrisers Hyderabad   
890  Royal Challengers Bangalore             Rajasthan Royals   
157          Chennai Super Kings             Rajasthan Royals   
386              Kings XI Punjab          Sunrisers Hyderabad   

                                                 Venue  \
336                                   Wankhede Stadium   
98                             Sharjah Cricket Stadium   
890                                           Newlands   
157       

In [ ]:
# info
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1629 entries, 0 to 1628
Data columns (total 18 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   title_x           1629 non-null   object 
 1   imdb_id           1629 non-null   object 
 2   poster_path       1526 non-null   object 
 3   wiki_link         1629 non-null   object 
 4   title_y           1629 non-null   object 
 5   original_title    1629 non-null   object 
 6   is_adult          1629 non-null   int64  
 7   year_of_release   1629 non-null   int64  
 8   runtime           1629 non-null   object 
 9   genres            1629 non-null   object 
 10  imdb_rating       1629 non-null   float64
 11  imdb_votes        1629 non-null   int64  
 12  story             1609 non-null   object 
 13  summary           1629 non-null   object 
 14  tagline           557 non-null    object 
 15  actors            1624 non-null   object 
 16  wins_nominations  707 non-null    object 


In [ ]:
ipl.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 950 entries, 0 to 949
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ID               950 non-null    int64  
 1   City             899 non-null    object 
 2   Date             950 non-null    object 
 3   Season           950 non-null    object 
 4   MatchNumber      950 non-null    object 
 5   Team1            950 non-null    object 
 6   Team2            950 non-null    object 
 7   Venue            950 non-null    object 
 8   TossWinner       950 non-null    object 
 9   TossDecision     950 non-null    object 
 10  SuperOver        946 non-null    object 
 11  WinningTeam      946 non-null    object 
 12  WonBy            950 non-null    object 
 13  Margin           932 non-null    float64
 14  method           19 non-null     object 
 15  Player_of_Match  946 non-null    object 
 16  Team1Players     950 non-null    object 
 17  Team2Players    

In [ ]:
# describe
movies.describe()

       is_adult  year_of_release  imdb_rating     imdb_votes
count    1629.0      1629.000000  1629.000000    1629.000000
mean        0.0      2010.263966     5.557459    5384.263352
std         0.0         5.381542     1.567609   14552.103231
min         0.0      2001.000000     0.000000       0.000000
25%         0.0      2005.000000     4.400000     233.000000
50%         0.0      2011.000000     5.600000    1000.000000
75%         0.0      2015.000000     6.800000    4287.000000
max         0.0      2019.000000     9.400000  310481.000000

In [ ]:
ipl.describe()

                 ID      Margin
count  9.500000e+02  932.000000
mean   8.304852e+05   17.056867
std    3.375678e+05   21.633109
min    3.359820e+05    1.000000
25%    5.012612e+05    6.000000
50%    8.297380e+05    8.000000
75%    1.175372e+06   19.000000
max    1.312200e+06  146.000000

In [ ]:
# isnull
movies.isnull().sum()

title_x                0
imdb_id                0
poster_path          103
wiki_link              0
title_y                0
original_title         0
is_adult               0
year_of_release        0
runtime                0
genres                 0
imdb_rating            0
imdb_votes             0
story                 20
summary                0
tagline             1072
actors                 5
wins_nominations     922
release_date         107
dtype: int64

In [ ]:
# duplicated
movies.duplicated().sum()

0

In [ ]:
students.duplicated().sum()

1

In [ ]:
# rename
students

    iq  percent  lpa
0  100       80   10
1   90       70    7
2  120      100   14
3   80       50    2
4    0        0    0
5    0        0    0

In [ ]:
students.rename(columns={'marks':'percent','package':'lpa'},inplace=True)

# 3. Math Methods and axis

## 3.1 The axis parameter

The maths methods learned for Series return a single scalar, because a Series has only one direction to collapse. A DataFrame has two, and the `axis` parameter chooses which one.

```python
students.sum(axis=0)
students.mean(axis=1)
students.var()
```

`axis=0` works DOWN the rows. The sum is computed per column, collapsing each column into one number, and the result is a Series indexed by column name. In the output, sum(axis=0) returns iq 390, percent 300, lpa 33 — three column totals. This is the default, so `students.var()` without an axis also works column-wise and returns one variance per column.

`axis=1` works ACROSS the columns. The mean is computed per row, collapsing each row into one number, and the result is a Series indexed by the row labels. Output shows a per-student average: row 0 near 63.33, row 1 near 55.67, row 2 at 78.0, and so on.

A solid memory hook: `axis=0` is vertical, so it drops the rows and leaves per-column results; `axis=1` is horizontal, so it drops the columns and leaves per-row results. Think of axis as the direction along which pandas moves, adding up everything it passes.

## 3.2 sum, mean, and var in context

Run the three computations against the actual output and the axis idea becomes concrete.

`sum(axis=0)` totals each column: the iq column adds to 390, percent to 300, lpa to 33. You can read the result as 'the whole-class totals for every attribute'.

`mean(axis=1)` averages each row: student 0 (100, 80, 10) averages to 63.33, student 2 (120, 100, 14) to 78.0, and the two students with all-zero rows average to 0.0. This is 'the average attribute score for each student' — a per-person summary.

`var()` (default axis 0) measures spread per column: iq has variance 2710.0, percent 1760.0, lpa 33.5. The high iq variance tells you class intelligence scores vary widely, while package variance is comparatively small.

Every aggregate — sum, mean, min, max, var, std — follows identical axis rules, and pandas automatically operates only on numeric columns when you run them, skipping text. Missing values are skipped by default (skipna), so a NaN in a column does not poison the total unless you ask for it. Sums for totals, means for centres, variances for spread: you now have the vocabulary to summarise any numeric part of a table.

In [ ]:
# sum -> axis argument
students.sum(axis=0)

iq         390
percent    300
lpa         33
dtype: int64

In [ ]:
students.mean(axis=1)

0    63.333333
1    55.666667
2    78.000000
3    44.000000
4     0.000000
5     0.000000
dtype: float64

In [ ]:
students.var()

iq         2710.0
percent    1760.0
lpa          33.5
dtype: float64

# 4. Selecting Columns

## 4.1 One column vs many

Column selection is the most common DataFrame operation, and pandas splits it into two distinct syntaxes.

```python
movies['title_x']
ipl['Venue']
```

Passing a single string in square brackets returns ONE column as a Series. The output shows the movie titles vertically with their index on the left, or the IPL venues — the Series keeps the DataFrame's row index, so you can still tell which row each value came from.

```python
movies[['year_of_release', 'actors', 'title_x']]
ipl[['Team1', 'Team2', 'WinningTeam']]
```

Passing a LIST of strings returns a new DataFrame containing exactly those columns, in the order you listed them. Nothing else is filtered — every row is present, and the result keeps the same index.

The rule of thumb: single brackets + string = Series; double brackets + list = DataFrame. The difference is the same as the difference between asking for one column of a table and asking for a smaller table. This distinction matters because a Series and a DataFrame support different subsequent operations, so knowing what you got back shapes what you can do next.

## 4.2 Deep dive on column selection

The examples in the notebook give you the full column-selection toolkit.

`movies['title_x']` returns the movie titles as a Series — one value per row, labels intact. `ipl['Venue']` does the same for the stadium names, and its output stretches over all 950 matches.

The multi-column examples build smaller DataFrames: `movies[['year_of_release', 'actors', 'title_x']]` gathers the year, the actor list, and the title side by side; `ipl[['Team1', 'Team2', 'WinningTeam']]` prepares a compact result frame. Column ORDER follows your list, and missing column names raise a KeyError — a helpful guard against typos.

Three extra facts complete the picture. First, the dot notation `ipl.Venue` also selects a column, but it can collide with DataFrame's own attribute names and reserved words, so brackets are preferred. Second, column selection is how you prepare inputs for analysis: grab the columns your question needs, then work on the slice. Third, creating brand-new columns uses the same bracket syntax on the left-hand side — `df['new'] = ...` — which the session demonstrates with the Country and lead actor columns. If you can comfortably choose one column or many, you have unlocked the door to every more advanced pandas pattern that follows.

In [ ]:
# single cols
movies['title_x']

0                   Uri: The Surgical Strike
1                              Battalion 609
2       The Accidental Prime Minister (film)
3                            Why Cheat India
4                            Evening Shadows
                        ...                 
1624                   Tera Mera Saath Rahen
1625                    Yeh Zindagi Ka Safar
1626                         Sabse Bada Sukh
1627                                   Daaka
1628                                Humsafar
Name: title_x, Length: 1629, dtype: object

In [ ]:
ipl['Venue']

0                Narendra Modi Stadium, Ahmedabad
1                Narendra Modi Stadium, Ahmedabad
2                           Eden Gardens, Kolkata
3                           Eden Gardens, Kolkata
4                        Wankhede Stadium, Mumbai
                          ...                    
945                                  Eden Gardens
946                              Wankhede Stadium
947                              Feroz Shah Kotla
948    Punjab Cricket Association Stadium, Mohali
949                         M Chinnaswamy Stadium
Name: Venue, Length: 950, dtype: object

In [ ]:
# multiple cols
movies[['year_of_release','actors','title_x']]

      year_of_release                                             actors  \
0                2019  Vicky Kaushal|Paresh Rawal|Mohit Raina|Yami Ga...   
1                2019  Vicky Ahuja|Shoaib Ibrahim|Shrikant Kamat|Elen...   
2                2019  Anupam Kher|Akshaye Khanna|Aahana Kumra|Atul S...   
3                2019  Emraan Hashmi|Shreya Dhanwanthary|Snighdadeep ...   
4                2018  Mona Ambegaonkar|Ananth Narayan Mahadevan|Deva...   
...               ...                                                ...   
1624             2001  Ajay Devgn|Sonali Bendre|Namrata Shirodkar|Pre...   
1625             2001  Ameesha Patel|Jimmy Sheirgill|Nafisa Ali|Gulsh...   
1626             2018  Vijay Arora|Asrani|Rajni Bala|Kumud Damle|Utpa...   
1627             2019                          Gippy Grewal|Zareen Khan|   
1628             2011                                        Fawad Khan|   

                                   title_x  
0                 Uri: The Surgical Strike

In [ ]:
ipl[['Team1','Team2','WinningTeam']]

                           Team1                        Team2  \
0               Rajasthan Royals               Gujarat Titans   
1    Royal Challengers Bangalore             Rajasthan Royals   
2    Royal Challengers Bangalore         Lucknow Super Giants   
3               Rajasthan Royals               Gujarat Titans   
4            Sunrisers Hyderabad                 Punjab Kings   
..                           ...                          ...   
945        Kolkata Knight Riders              Deccan Chargers   
946               Mumbai Indians  Royal Challengers Bangalore   
947             Delhi Daredevils             Rajasthan Royals   
948              Kings XI Punjab          Chennai Super Kings   
949  Royal Challengers Bangalore        Kolkata Knight Riders   

                     WinningTeam  
0                 Gujarat Titans  
1               Rajasthan Royals  
2    Royal Challengers Bangalore  
3                 Gujarat Titans  
4                   Punjab Kings  
..       

# 5. Selecting Rows — iloc and loc

## 5.1 Position with iloc, label with loc

Rows are selected with two accessors that make the POSITION-vs-LABEL choice explicit.

```python
movies.iloc[5]
movies.iloc[:5]
movies.iloc[[0, 4, 5]]
```

`iloc` works by integer POSITION, exactly like list indexing. `movies.iloc[5]` returns the whole 6th row (position 5) as a Series with columns as its index. `movies.iloc[:5]` slices the first five rows into a DataFrame, and `movies.iloc[[0, 4, 5]]` grabs rows at positions 0, 4, and 5 — fancy indexing. Positions are 0-based and the slice's stop is exclusive, matching NumPy and Python lists.

```python
students.loc['nitish']
students.loc['nitish':'rishabh':2]
students.loc[['nitish', 'ankita', 'rupesh']]
```

`loc` works by INDEX LABEL. `students.loc['nitish']` pulls out the row whose name label is nitish. `students.loc['nitish':'rishabh':2]` slices from the nitish label to the rishabh label, stepping by two, and — crucially — label slices are INCLUSIVE of the end label, unlike positional slices. `students.loc[['nitish', 'ankita', 'rupesh']]` selects three rows by their labels in one go.

The decision rule: use iloc when you know the row number, loc when you know the row's name.

## 5.2 Deep dive on row access

Run through the actual outputs to cement the two accessors.

`movies.iloc[5]` returns a Series titled `Soni (film)`: title_x, imdb_id, poster_path, and every other column becomes a label in that Series, with that movie's values under them. A single-row iloc/loc call therefore returns a Series; slices and lists return DataFrames.

`movies.iloc[:5]` and `movies.iloc[[0, 4, 5]]` return the DataFrame rows 0-4 and rows 0, 4, 5 respectively. `students.iloc[[0, 3, 4]]` shows the positional way to grab nitish, rishabh, and amit — those rows happen to sit at positions 0, 3, 4.

The loc examples use the name-based index set earlier. `students.loc['nitish']` shows the single student's iq, marks, and package. `students.loc['nitish':'rishabh':2]` keeps only the labels nitish and rupesh because stepping by 2 over four names skips ankit and rishabh. `students.loc[['nitish', 'ankita', 'rupesh']]` reorders to exactly the three requested.

Two professional habits close the section. When you need ONE cell fast, `.at[label]` and `.iat[position]` skip pandas' ambiguity checks entirely. And never mix the two styles in one expression — a label looked up positionally, or vice versa, is the classic source of silent wrong answers.

In [ ]:
# single row
movies.iloc[5]

title_x                                                   Soni (film)
imdb_id                                                     tt6078866
poster_path         https://upload.wikimedia.org/wikipedia/en/thum...
wiki_link                   https://en.wikipedia.org/wiki/Soni_(film)
title_y                                                          Soni
original_title                                                   Soni
is_adult                                                            0
year_of_release                                                  2018
runtime                                                            97
genres                                                          Drama
imdb_rating                                                       7.2
imdb_votes                                                       1595
story               Soni  a young policewoman in Delhi  and her su...
summary             While fighting crimes against women in Delhi  ...
tagline             

In [ ]:
# multiple row
movies.iloc[:5]

                                title_x    imdb_id  \
0              Uri: The Surgical Strike  tt8291224   
1                         Battalion 609  tt9472208   
2  The Accidental Prime Minister (film)  tt6986710   
3                       Why Cheat India  tt8108208   
4                       Evening Shadows  tt6028796   

                                         poster_path  \
0  https://upload.wikimedia.org/wikipedia/en/thum...   
1                                                NaN   
2  https://upload.wikimedia.org/wikipedia/en/thum...   
3  https://upload.wikimedia.org/wikipedia/en/thum...   
4                                                NaN   

                                           wiki_link  \
0  https://en.wikipedia.org/wiki/Uri:_The_Surgica...   
1        https://en.wikipedia.org/wiki/Battalion_609   
2  https://en.wikipedia.org/wiki/The_Accidental_P...   
3      https://en.wikipedia.org/wiki/Why_Cheat_India   
4      https://en.wikipedia.org/wiki/Evening_Shadows   

 

In [ ]:
# fancy indexing
movies.iloc[[0,4,5]]

                    title_x    imdb_id  \
0  Uri: The Surgical Strike  tt8291224   
4           Evening Shadows  tt6028796   
5               Soni (film)  tt6078866   

                                         poster_path  \
0  https://upload.wikimedia.org/wikipedia/en/thum...   
4                                                NaN   
5  https://upload.wikimedia.org/wikipedia/en/thum...   

                                           wiki_link  \
0  https://en.wikipedia.org/wiki/Uri:_The_Surgica...   
4      https://en.wikipedia.org/wiki/Evening_Shadows   
5          https://en.wikipedia.org/wiki/Soni_(film)   

                    title_y            original_title  is_adult  \
0  Uri: The Surgical Strike  Uri: The Surgical Strike         0   
4           Evening Shadows           Evening Shadows         0   
5                      Soni                      Soni         0   

   year_of_release runtime            genres  imdb_rating  imdb_votes  \
0             2019     138  Action|Dram

In [ ]:
# loc
students

          iq  marks  package
name                        
nitish   100     80       10
ankit     90     70        7
rupesh   120    100       14
rishabh   80     50        2
amit       0      0        0
ankita     0      0        0

In [ ]:
students.loc['nitish']

iq         100
marks       80
package     10
Name: nitish, dtype: int64

In [ ]:
students.loc['nitish':'rishabh':2]

         iq  marks  package
name                       
nitish  100     80       10
rupesh  120    100       14

In [ ]:
students.loc[['nitish','ankita','rupesh']]

         iq  marks  package
name                       
nitish  100     80       10
ankita    0      0        0
rupesh  120    100       14

In [ ]:
students.iloc[[0,3,4]]

          iq  marks  package
name                        
nitish   100     80       10
rishabh   80     50        2
amit       0      0        0

# 6. Selecting Rows and Columns Together

## 6.1 Two-dimensional selection

The two accessors combine in a single expression when you need a rectangular window: rows first, then columns.

```python
movies.iloc[0:3, 0:3]
movies.loc[0:2, 'title_x':'poster_path']
```

`movies.iloc[0:3, 0:3]` reads as 'rows at positions 0 to 2, columns at positions 0 to 2'. The result is the top-left 3 by 3 corner of the table: the titles, imdb ids, and poster paths of the first three movies. Positional slicing remains half-open, so 0:3 means positions 0, 1, and 2.

`movies.loc[0:2, 'title_x':'poster_path']` uses labels on BOTH axes. Rows 0 to 2 inclusive, columns from title_x up to and including poster_path — label slices are inclusive at both ends, which is why poster_path makes it into this result even though it sits three columns to the right of title_x.

The comma is the whole trick: everything before it selects rows, everything after it selects columns. Since the movies frame keeps its original RangeIndex, both the positional and label styles happen to pick the same window here — a coincidence that disappears the moment you set a custom index. The result of either call is a small DataFrame containing exactly the intersecting cells.

## 6.2 Deep dive on two-dimensional selection

These two calls look nearly identical and land on the same window — and that is precisely why they are a great teaching pair.

`iloc[0:3, 0:3]` is pure position. The code does not care what the columns are called; it takes the first three rows and the first three columns (positions 0 to 2). If the column order ever changes, this slice silently grabs different columns.

`loc[0:2, 'title_x':'poster_path']` is pure label. It starts from whatever label any integer 0, 1, 2 actually names in the index, and it walks the columns by NAME from title_x to poster_path. If you reordered the columns, the same labels would be found wherever they moved.

Both styles accept the full range of selectors you already know: single values, slices, step slices, and lists. So `movies.iloc[[0, 5], [1, 3]]` (positional lists) and `movies.loc[0:2, ['title_x', 'imdb_id']]` (label list) work exactly as expected.

Chopping a precise two-dimensional window out of a table is the heart of data preparation — feed the window to a chart, a model, or another analysis step. Choosing iloc or loc comes down to one question: do you trust positions or names? Use whichever the data makes trustworthy, and the same rectangular-window logic will serve you through the whole course.

In [ ]:
movies.iloc[0:3,0:3]

                                title_x    imdb_id  \
0              Uri: The Surgical Strike  tt8291224   
1                         Battalion 609  tt9472208   
2  The Accidental Prime Minister (film)  tt6986710   

                                         poster_path  
0  https://upload.wikimedia.org/wikipedia/en/thum...  
1                                                NaN  
2  https://upload.wikimedia.org/wikipedia/en/thum...  

In [ ]:
movies.loc[0:2,'title_x':'poster_path']

                                title_x    imdb_id  \
0              Uri: The Surgical Strike  tt8291224   
1                         Battalion 609  tt9472208   
2  The Accidental Prime Minister (film)  tt6986710   

                                         poster_path  
0  https://upload.wikimedia.org/wikipedia/en/thum...  
1                                                NaN  
2  https://upload.wikimedia.org/wikipedia/en/thum...  

# 7. Filtering Rows with Conditions

In [ ]:
ipl.head(2)

        ID       City        Date Season  MatchNumber  \
0  1312200  Ahmedabad  2022-05-29   2022        Final   
1  1312199  Ahmedabad  2022-05-27   2022  Qualifier 2   

                         Team1             Team2  \
0             Rajasthan Royals    Gujarat Titans   
1  Royal Challengers Bangalore  Rajasthan Royals   

                              Venue        TossWinner TossDecision SuperOver  \
0  Narendra Modi Stadium, Ahmedabad  Rajasthan Royals          bat         N   
1  Narendra Modi Stadium, Ahmedabad  Rajasthan Royals        field         N   

        WinningTeam    WonBy  Margin method Player_of_Match  \
0    Gujarat Titans  Wickets     7.0    NaN       HH Pandya   
1  Rajasthan Royals  Wickets     7.0    NaN      JC Buttler   

                                        Team1Players  \
0  ['YBK Jaiswal', 'JC Buttler', 'SV Samson', 'D ...   
1  ['V Kohli', 'F du Plessis', 'RM Patidar', 'GJ ...   

                                        Team2Players      Umpire1      

## 7.1 Boolean masks

Column selection picks columns; filtering picks ROWS by a condition. The building block is the boolean mask.

```python
mask = ipl['MatchNumber'] == 'Final'
new_df = ipl[mask]
```

`ipl['MatchNumber'] == 'Final'` compares every row's MatchNumber against the string 'Final' and produces a boolean Series — True where the match was a final, False otherwise. Passing that mask inside the brackets keeps only the True rows. The subsequent line, `new_df[['Season', 'WinningTeam']]`, selects two columns from the filtered frame, and the combined output lists every final with its season and winner.

The same one-liner style covers multiple conditions with the boolean operators `&` (and), `|` (or), and `~` (not):

```python
ipl[(ipl['City'] == 'Kolkata') & (ipl['WinningTeam'] == 'Chennai Super Kings')]
movies[(movies['imdb_rating'] > 8.5) & (movies['vote_count'] > 10000)]
```

Parentheses around EVERY condition are mandatory — without them, Python's operator precedence pulls the comparisons apart. The result is the pandas equivalent of SQL's WHERE clause.

## 7.2 Deep dive on filtering

The notebook answers five real cricket and cinema questions with masks; trace each one.

`ipl[ipl['MatchNumber'] == 'Final'][['Season', 'WinningTeam']]` collapses the 'find all final winners' question into one expression: filter, then choose columns. `ipl[ipl['SuperOver'] == 'Y'].shape[0]` counts the super-over finishes — 14. `ipl[(ipl['City'] == 'Kolkata') & (ipl['WinningTeam'] == 'Chennai Super Kings')].shape[0]` reports 5 wins. The toss question compares two columns at once: `ipl[ipl['TossWinner'] == ipl['WinningTeam']]` keeps rows where the toss-winner and the match-winner are the same team, and the cell computes that proportion as 51.47 percent.

The cinema filter is the strictest: `movies[(movies['imdb_rating'] > 8.5) & (movies['imdb_votes'] > 10000)].shape[0]` returns 0 — no bollywood movie meets both bars, which is itself a finding.

The Action-genre question shows how masks compose with string tools: `mask1 = movies['genres'].str.contains('Action')` flags genres containing the word Action, `mask2 = movies['imdb_rating'] > 7.5` sets the rating bar, and `movies[mask1 & mask2]` returns the intersection. A commented alternative splits genres and checks membership with apply — same goal, two routes. Storing masks in named variables keeps multi-condition filters readable.

In [ ]:
# find all the final winners
mask = ipl['MatchNumber'] == 'Final'
new_df = ipl[mask]
new_df[['Season','WinningTeam']]

ipl[ipl['MatchNumber'] == 'Final'][['Season','WinningTeam']]

      Season            WinningTeam
0       2022         Gujarat Titans
74      2021    Chennai Super Kings
134  2020/21         Mumbai Indians
194     2019         Mumbai Indians
254     2018    Chennai Super Kings
314     2017         Mumbai Indians
373     2016    Sunrisers Hyderabad
433     2015         Mumbai Indians
492     2014  Kolkata Knight Riders
552     2013         Mumbai Indians
628     2012  Kolkata Knight Riders
702     2011    Chennai Super Kings
775  2009/10    Chennai Super Kings
835     2009        Deccan Chargers
892  2007/08       Rajasthan Royals

In [ ]:
# how many super over finishes have occured
ipl[ipl['SuperOver'] == 'Y'].shape[0]

14

In [ ]:
# how many matches has csk won in kolkata
ipl[(ipl['City'] == 'Kolkata') & (ipl['WinningTeam'] == 'Chennai Super Kings')].shape[0]

5

In [ ]:
# toss winner is match winner in percentage
(ipl[ipl['TossWinner'] == ipl['WinningTeam']].shape[0]/ipl.shape[0])*100

51.473684210526315

In [ ]:
# movies with rating higher than 8 and votes>10000
movies[(movies['imdb_rating'] > 8.5) & (movies['imdb_votes'] > 10000)].shape[0]

0

In [ ]:
# Action movies with rating higher than 7.5
# mask1 = movies['genres'].str.split('|').apply(lambda x:'Action' in x)
mask1 = movies['genres'].str.contains('Action')
mask2 = movies['imdb_rating'] > 7.5

movies[mask1 & mask2]

                            title_x    imdb_id  \
0          Uri: The Surgical Strike  tt8291224   
41             Family of Thakurganj  tt8897986   
84                        Mukkabaaz  tt7180544   
106                           Raazi  tt7098658   
110   Parmanu: The Story of Pokhran  tt6826438   
112         Bhavesh Joshi Superhero  tt6129302   
169                The Ghazi Attack  tt6299040   
219                Raag Desh (film)  tt6080746   
258                   Irudhi Suttru  tt5310090   
280                       Laal Rang  tt5600714   
297                     Udta Punjab  tt4434004   
354                   Dangal (film)  tt5074352   
362               Bajrangi Bhaijaan  tt3863552   
365          Baby (2015 Hindi film)  tt3848892   
393      Detective Byomkesh Bakshy!  tt3447364   
449               Titli (2014 film)  tt3019620   
536                   Haider (film)  tt3390572   
589                    Vishwaroopam  tt2199711   
625                     Madras Cafe  tt2855648   


In [ ]:
# write a function that can return the track record of 2 teams against each other

# 8. Adding New Columns

## 8.1 Constant and derived columns

New columns arrive through assignment, exactly like adding a key to a dictionary.

```python
movies['Country'] = 'India'
```

Assigning a scalar to a brand-new column broadcasts that value to every row: the entire column instantly reads 'India'. This is the constant column — the fastest way to stamp a label onto a whole table.

```python
movies['lead actor'] = movies['actors'].str.split('|').apply(lambda x: x[0])
```

The derived column computes new values from existing ones. Reading left to right: `movies['actors']` takes the actor list column, `.str` opens pandas' string accessor, `.split('|')` splits each row's text into a list at the pipe character, `.apply(lambda x: x[0])` takes the first element of each list, and the whole expression is assigned to a fresh column named 'lead actor'. The leading actor is now its own column, ready for its own filtering and counting.

Notice that `movies.dropna(inplace=True)` runs first in the notebook, removing the rows with missing values so the split-based derivation does not trip over empty text. Clean, then derive: that ordering is a real-world habit, not just a convenience.

## 8.2 Deep dive on new columns

The three-column workflow in this section tells a complete story about feature engineering.

First, the constant: `movies['Country'] = 'India'` produces a column where every one of the 298 remaining movies shares the same value. After the earlier dropna and the head print, the Country column tiles the frame. Constants are trivial, but they matter when a dataset must carry a fixed attribute — the country, the currency, the dataset version.

Second, the derived: the lead-actor column parses the pipe-separated `actors` field with `.str.split('|')` and keeps the first name with `.apply(lambda x: x[0])`. The printed head shows Gully Boy, Yeh Hai India, Article 15, Aiyaary, and Raid with their respective lead actors extracted. The original actors column remains untouched; the new column is added and positioned at the right edge of the table.

Third, verification: `movies.info()` after the additions reports 19 columns (up from 18) and still holds the rows survived by dropna. The info chart is the receipt that the column was really created.

The general pattern — `df['new'] = expression` — applies to all three: a scalar broadcasts a constant, a Series aligns its rows by index, and any per-row computation through apply or the .str tools constructs values from your existing features. Feature engineering is simply this pattern repeated many times.

In [ ]:
# completely new
movies['Country'] = 'India'
movies.head()

                                title_x    imdb_id  \
0              Uri: The Surgical Strike  tt8291224   
1                         Battalion 609  tt9472208   
2  The Accidental Prime Minister (film)  tt6986710   
3                       Why Cheat India  tt8108208   
4                       Evening Shadows  tt6028796   

                                         poster_path  \
0  https://upload.wikimedia.org/wikipedia/en/thum...   
1                                                NaN   
2  https://upload.wikimedia.org/wikipedia/en/thum...   
3  https://upload.wikimedia.org/wikipedia/en/thum...   
4                                                NaN   

                                           wiki_link  \
0  https://en.wikipedia.org/wiki/Uri:_The_Surgica...   
1        https://en.wikipedia.org/wiki/Battalion_609   
2  https://en.wikipedia.org/wiki/The_Accidental_P...   
3      https://en.wikipedia.org/wiki/Why_Cheat_India   
4      https://en.wikipedia.org/wiki/Evening_Shadows   

 

In [ ]:
# from existing ones
movies.dropna(inplace=True)

In [ ]:
movies['lead actor'] = movies['actors'].str.split('|').apply(lambda x:x[0])
movies.head()

              title_x     imdb_id  \
11          Gully Boy   tt2395469   
34      Yeh Hai India   tt5525846   
37  Article 15 (film)  tt10324144   
87            Aiyaary   tt6774212   
96   Raid (2018 film)   tt7363076   

                                          poster_path  \
11  https://upload.wikimedia.org/wikipedia/en/thum...   
34  https://upload.wikimedia.org/wikipedia/en/thum...   
37  https://upload.wikimedia.org/wikipedia/en/thum...   
87  https://upload.wikimedia.org/wikipedia/en/thum...   
96  https://upload.wikimedia.org/wikipedia/en/thum...   

                                          wiki_link        title_y  \
11          https://en.wikipedia.org/wiki/Gully_Boy      Gully Boy   
34      https://en.wikipedia.org/wiki/Yeh_Hai_India  Yeh Hai India   
37  https://en.wikipedia.org/wiki/Article_15_(film)     Article 15   
87            https://en.wikipedia.org/wiki/Aiyaary        Aiyaary   
96   https://en.wikipedia.org/wiki/Raid_(2018_film)           Raid   

   original_t

In [ ]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 298 entries, 11 to 1623
Data columns (total 19 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   title_x           298 non-null    object 
 1   imdb_id           298 non-null    object 
 2   poster_path       298 non-null    object 
 3   wiki_link         298 non-null    object 
 4   title_y           298 non-null    object 
 5   original_title    298 non-null    object 
 6   is_adult          298 non-null    int64  
 7   year_of_release   298 non-null    int64  
 8   runtime           298 non-null    object 
 9   genres            298 non-null    object 
 10  imdb_rating       298 non-null    float64
 11  imdb_votes        298 non-null    int64  
 12  story             298 non-null    object 
 13  summary           298 non-null    object 
 14  tagline           298 non-null    object 
 15  actors            298 non-null    object 
 16  wins_nominations  298 non-null    object 


# 9. Dtype Optimization

## 9.1 astype and category

DataFrames often carry columns whose dtype is far fatter than the data requires. This section shows two ways to slim them down.

```python
ipl['ID'] = ipl['ID'].astype('int32')
```

The integer match IDs are stored as int64 by default. `astype('int32')` downcasts them to a 32-bit integer, halving the bytes each ID occupies. The info() snapshot before and after makes the change visible: the ID row's dtype flips from int64 to int32.

```python
ipl['Team1'] = ipl['Team1'].astype('category')
ipl['Team2'] = ipl['Team2'].astype('category')
```

Team names are the bigger win. A column like Team1 repeats the same handful of franchise names thousands of times, and storing each occurrence as a full text string wastes memory. The category dtype stores each unique name ONCE, in a small catalogue, and internally records each row as a compact code pointing at that catalogue. The printed info() reflects the change on the Team1 and Team2 rows.

The lesson generalises: assign `df['col'] = df['col'].astype(...)` to convert in place, and reach for category whenever a column holds many repeated strings — team names, cities, genres. The memory savings compound quickly on large data.

## 9.2 Deep dive on dtype

The comparison is best seen in the info() output, which prints a before-and-after report for the IPL frame.

Initially every column is either int64 (ID) or object (everything else). Object columns are the pandas catch-all for text, but they are the slowest and heaviest dtype available. After the conversions, ID reads int32, and Team1 and Team2 read category — and the structure of a big notebook like this is exactly where the savings start to matter.

Why does category help so much? Think about the Team1 column: 950 IPL matches, but only a handful of distinct franchise names. A category column keeps each name once in a coding table and stores codes — tiny integers — in the 950 rows. The full names are recovered on demand, and operations like value_counts become blazingly fast because pandas can count codes instead of comparing strings.

Category also offers ordering semantics: categories have a natural order that pandas can sort by without alphabetical coercion, and you can even specify the order of categories when you build the dtype. Those details belong to a later session, however. For now, hold two actions in mind: `astype` changes a column's dtype to something narrower, and `category` is the special dtype for repeated labels. Laziness about dtypes is the most common cause of both inflated memory and slow pipelines on real datasets.

In [ ]:
# astype
ipl.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 950 entries, 0 to 949
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ID               950 non-null    int64  
 1   City             899 non-null    object 
 2   Date             950 non-null    object 
 3   Season           950 non-null    object 
 4   MatchNumber      950 non-null    object 
 5   Team1            950 non-null    object 
 6   Team2            950 non-null    object 
 7   Venue            950 non-null    object 
 8   TossWinner       950 non-null    object 
 9   TossDecision     950 non-null    object 
 10  SuperOver        946 non-null    object 
 11  WinningTeam      946 non-null    object 
 12  WonBy            950 non-null    object 
 13  Margin           932 non-null    float64
 14  method           19 non-null     object 
 15  Player_of_Match  946 non-null    object 
 16  Team1Players     950 non-null    object 
 17  Team2Players    

In [ ]:
ipl['ID'] = ipl['ID'].astype('int32')

In [ ]:
ipl.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 950 entries, 0 to 949
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ID               950 non-null    int32  
 1   City             899 non-null    object 
 2   Date             950 non-null    object 
 3   Season           950 non-null    object 
 4   MatchNumber      950 non-null    object 
 5   Team1            950 non-null    object 
 6   Team2            950 non-null    object 
 7   Venue            950 non-null    object 
 8   TossWinner       950 non-null    object 
 9   TossDecision     950 non-null    object 
 10  SuperOver        946 non-null    object 
 11  WinningTeam      946 non-null    object 
 12  WonBy            950 non-null    object 
 13  Margin           932 non-null    float64
 14  method           19 non-null     object 
 15  Player_of_Match  946 non-null    object 
 16  Team1Players     950 non-null    object 
 17  Team2Players    

In [ ]:
# ipl['Season'] = ipl['Season'].astype('category')
ipl['Team1'] = ipl['Team1'].astype('category')
ipl['Team2'] = ipl['Team2'].astype('category')

In [ ]:
ipl.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 950 entries, 0 to 949
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   ID               950 non-null    int32   
 1   City             899 non-null    object  
 2   Date             950 non-null    object  
 3   Season           950 non-null    category
 4   MatchNumber      950 non-null    object  
 5   Team1            950 non-null    category
 6   Team2            950 non-null    category
 7   Venue            950 non-null    object  
 8   TossWinner       950 non-null    object  
 9   TossDecision     950 non-null    object  
 10  SuperOver        946 non-null    object  
 11  WinningTeam      946 non-null    object  
 12  WonBy            950 non-null    object  
 13  Margin           932 non-null    float64 
 14  method           19 non-null     object  
 15  Player_of_Match  946 non-null    object  
 16  Team1Players     950 non-null    object  
 1

## 9.3 value_counts for exploration

The section closes with the exploration tool that pairs perfectly with categorical columns.

```python
ipl['Player_of_Match'].value_counts()
```

`value_counts()` counts how many times each distinct value appears and returns a Series sorted descending. Which player won player-of-the-match most often? The method answers it instantly, and the same call works on TossDecision or Season.

The notebook's commented starter questions give you the flavour of real analysis:

- Which player won the most player-of-the-match awards in finals and qualifiers? Filter those rows first, then value_counts.
- Plot the toss-decision distribution: `ipl['TossDecision'].value_counts().plot(kind='bar')`.
- Count how many matches each team has played: `ipl['Team1'].value_counts() + ipl['Team2'].value_counts()` — add Team1 and Team2 counts so both home and away appearances are counted.

This is the same value_counts you met on Series, now pointed at DataFrame columns. Combined with astype-category columns, it becomes the fastest frequency analysis available in pandas: one call turns any categorical column into a sorted popularity report, and a plot call turns that report into a picture. Use it as your standard answer whenever a question begins with the words 'how many of each'.

In [ ]:
# value_counts

In [ ]:
# find which player has won most potm -> in finals and qualifiers

In [ ]:
# Toss decision plot

In [ ]:
# how many matches each team has played

# Summary

In this session you turned the one-dimensional Series into the two-dimensional DataFrame. You built tables from lists of lists, from dictionaries of lists, and directly from CSV files with read_csv, and you inspected them with shape, dtypes, index, columns, values, info, and describe - adding isnull, duplicated, and rename to the health check.

You ran maths across rows and columns with the axis parameter, selected single columns into Series and several into DataFrames, and selected rows by position with iloc and by label with loc, including two-dimensional windows. Boolean masks filtered rows with combined conditions - the pandas WHERE clause. New columns arrived as constants or derived values, astype and category slimmed memory, and value_counts produced instant frequency reports.

In [ ]:
# sort_values -> ascending -> na_position -> inplace -> multiple cols